In [ ]:
# ANNA LYN WILLIAMS
# Computationally Modelling The Go / No-Go Task in Depression
# Niv Lab Rotation
# 2026


import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.optimize import minimize

# TASK STRUCTURE

robots = {
    "GW":   {"correct": "go"},
    "NGW":  {"correct": "nogo"},
    "GAL":  {"correct": "go"},
    "NGAL": {"correct": "nogo"},
}

win_robots  = ["GW", "NGW"]
loss_robots = ["GAL", "NGAL"]

# 30 trials per robot per block, 2 blocks per timepoint
# 3 runes per robot per block, with 8/10/12 trials each (counterbalanced)
trials_per_robot_per_block  = 30
trials_per_robot            = trials_per_robot_per_block * 2  # 60 total
n_runes_per_robot_per_block = 3
min_trials_per_rune         = 8   # minimum trials any rune gets; used for aligned averaging


def sigmoid(x):
    return 1 / (1 + np.exp(-x))



# PREPROCESS DATA

def build_rune_map(df):
    """
    For each subject, map every rune letter (e.g. "L", "U", etc.) to its robot type (e.g. "GW", "NGW", etc.).

    Returns
    -------
    rune_map : dict  subject → {rune_letter: robot_name}
    """
    rune_map = {}

    for subj, subj_df in df.groupby("subject"):   # iterate one subject at a time

        rune_map[subj] = (
            subj_df[["rune", "robot"]]             # keep only the two relevant columns
            .drop_duplicates()                     # one row per rune (same mapping repeats every trial)
            .set_index("rune")                     # rune letter becomes the index
            ["robot"]                              # select robot column → Series: rune → robot
            .to_dict()                             # convert to plain dict: {"A": "GW", "B": "NGW", ...}
        )

    return rune_map                                # rune_map[subj]["A"] → "GW"


def load_real_data_with_df(csv_path):
    """
    Load a single-timepoint CSV. Returns:
      - actions  : dict  robot → (n_subjects, 60)  — block1 trials first, then block2
      - rewards  : dict  robot → (n_subjects, 60)
      - subjects : np.array of subject IDs
      - rune_map : dict  subject → {rune: robot}
      - df       : cleaned DataFrame (needed for build_rune_data)
    """
    # ---- load & clean --------------------------------------------------

    df = pd.read_csv(csv_path)

    df["robot"] = df["robot"].str.upper()                    # "gw" → "GW" so it matches the robots dict
    df["rune"]  = df["rune"].astype(str).str.strip().str.upper()  # strip whitespace, uppercase rune letters
    df["a"]     = (df["choice"] == " ").astype(int)          # go = space-bar press → 1; nogo = no press → 0
    df["r"] = df["outcome"].astype(float)   # raw outcomes: 10=better reward in win domain, 1=worse reward in win domain, -1=better reward in loss domain, -10=worse reward in loss domain
    
        # NOTE: robots will be recoded to 0/1 in build_rune_data: Paper: r = 1 for the BETTER outcome, r = 0 for the WORSE outcome,
    
        # within each valence domain separately.

    # ---- subject list --------------------------------------------------

    subjects   = df["subject"].unique()   # ordered array of unique subject IDs as they appear in the CSV
    n_subjects = len(subjects)

    # ---- pre-allocate output arrays ------------------------------------

    # For each robot, one row per subject, 60 columns (30 trials block1 + 30 trials block2)
    actions = {rob: np.zeros((n_subjects, trials_per_robot)) for rob in robots}
    rewards = {rob: np.zeros((n_subjects, trials_per_robot)) for rob in robots}

    # ---- fill arrays ---------------------------------------------------

    for s_idx, subj in enumerate(subjects):

        # All rows for this subject, sorted so block1 trials come before block2
        # (guarantees the 60-column layout is always block1-first regardless of CSV order)
        subj_df = df[df["subject"] == subj].sort_values(["block", "trial"])

        for rob in robots:
            # All trials for this subject + robot across both blocks
            rob_df = subj_df[subj_df["robot"] == rob].reset_index(drop=True)

            # Write into row s_idx of the pre-allocated array;
            # [:trials_per_robot] guards against any extra rows in the CSV
            actions[rob][s_idx, :] = rob_df["a"].values[:trials_per_robot]
            rewards[rob][s_idx, :] = rob_df["r"].values[:trials_per_robot]

    # ---- rune → robot lookup table ------------------------------------

    rune_map = build_rune_map(df)   # {subj: {rune_letter: robot_name}}

    return actions, rewards, subjects, rune_map, df


def build_rune_data(df, subjects, pit_ids=None):
    """
    Build per-subject, per-block, per-rune data structure.

    Structure:
        rune_data[s_idx][block][robot][rune] = {
            "actions": (1, n_trials),   # n_trials is 8, 10, or 12
            "rewards": (1, n_trials),   # recoded to 0/1 per paper convention
        }

    Reward recoding (applied here, before storing):
        Raw outcome values from the CSV use arbitrary point magnitudes:
            Win  robots (GW, NGW):   10 = win (better),        1 = no-win (worse)
            Loss robots (GAL, NGAL): -1 = avoided loss (better), -10 = received loss (worse)

        The paper recodes all outcomes to r = 1 (better) / r = 0 (worse) within
        each valence domain separately. This is mathematically equivalent to using
        the raw values because:
          (a) Q-values are learned separately for win and loss robots, so there is
              no cross-domain comparison that the scale could distort.
          (b) The softmax / sigmoid choice function is invariant to additive constants,
              so action probabilities derived from Q-values of, say, 0 and 1 are
              identical to those derived from -10 and -1 (same relative difference).
        Without recoding, raw values like 10 or -10 produce prediction errors up to
        ±9.5 on the first trial (vs. the intended ±0.5), which causes Q-values to
        overshoot their [0, 1] range and destabilises the fit — especially for loss robots.

    Subjects with unexpected rune counts are excluded and reported.
    If pit_ids is provided, only subjects in that list are processed.

    Returns
    -------
    rune_data         : list of dicts, one per complete subject
    complete_subjects : np.array of kept subject IDs
    excluded_subjects : np.array of dropped subject IDs
    """

    # ---- optional PIT filter -------------------------------------------
    # Restrict to PIT-approved subjects before doing any work,
    # so the loop below never processes subjects we don't need.

    if pit_ids is not None:
        n_before  = len(subjects)
        subjects  = subjects[np.isin(subjects, pit_ids)]
        print(f"  PIT filter: {len(subjects)} / {n_before} subjects kept")

    # ---- output containers ---------------------------------------------

    rune_data         = []
    complete_subjects = []
    excluded_subjects = []

    # ---- iterate over subjects -----------------------------------------

    for subj in subjects:

        # All rows for this subject, sorted so trials are in chronological order
        subj_df   = df[df["subject"] == subj].sort_values(["block", "trial"])
        subj_dict = {}    # will hold the nested block → robot → rune structure for this subject
        complete  = True  # flag; set to False if any rune count check fails

        # ---- iterate over blocks ---------------------------------------

        for block in [1, 2]:

            block_df         = subj_df[subj_df["block"] == block]   # rows for this block only
            subj_dict[block] = {}                                    # initialise block-level dict

            # ---- iterate over robots -----------------------------------

            for robot in robots:

                rob_df                  = block_df[block_df["robot"] == robot]   # rows for this robot in this block
                subj_dict[block][robot] = {}                                     # initialise robot-level dict

                runes = rob_df["rune"].unique()   # which rune letters appeared for this robot in this block

                # Sanity check: every robot should have exactly 3 runes per block.
                # If not, this subject's data is malformed — flag and break out of all loops.
                if len(runes) != n_runes_per_robot_per_block:
                    print(f"  EXCLUDED {subj}: block={block} robot={robot} "
                          f"has {len(runes)} runes (expected {n_runes_per_robot_per_block})")
                    complete = False
                    break   # break out of robot loop

                # ---- iterate over runes --------------------------------

                for rune in runes:

                    # All trials for this specific rune (8, 10, or 12 rows)
                    rune_df = rob_df[rob_df["rune"] == rune].reset_index(drop=True)

                    a_vals = rune_df["a"].values                # binary: 1 = go, 0 = nogo
                    r_vals = rune_df["r"].values.copy()         # copy so we don't modify the DataFrame

                    # ---- recode raw outcomes to Samuel & Gili's paper 0/1 convention ----
                    if robot in win_robots:
                        # Raw:     10 = win (better outcome),   1 = no-win (worse outcome)
                        # Recoded:  1 = win (better outcome),   0 = no-win (worse outcome)
                        r_vals = np.where(r_vals == 10, 1, 0) # since win robots can only be 10 or 1, this maps 10→1 and everything else (1) → 0

                    else:   # loss robots: GAL, NGAL
                        # Raw:     -1 = avoided loss (better outcome),  -10 = received loss (worse outcome)
                        # Recoded:  1 = avoided loss (better outcome),    0 = received loss (worse outcome)
                        r_vals = np.where(r_vals == -1, 1, 0) # since loss robots can only be -1 or -10, this maps -1→1 and everything else (-10) → 0

                    # Store as (1, n_trials) arrays — the leading 1 dimension makes them
                    # compatible with neg_log_likelihood, which indexes the sequence via arrays[0]
                    subj_dict[block][robot][rune] = {
                        "actions": a_vals.reshape(1, -1),
                        "rewards": r_vals.reshape(1, -1),   # recoded to 0/1 per paper convention
                    }

                if not complete:
                    break   # break out of rune loop (rune count check failed above)
            if not complete:
                break       # break out of block loop

        # ---- store or discard this subject -----------------------------

        if complete:
            rune_data.append(subj_dict)      # add nested dict to output list
            complete_subjects.append(subj)
        else:
            excluded_subjects.append(subj)   # log but don't add to rune_data

    # ---- convert to arrays for easy indexing later ---------------------

    complete_subjects = np.array(complete_subjects)
    excluded_subjects = np.array(excluded_subjects)

    print(f"  Kept {len(complete_subjects)} / {len(subjects)} subjects "
          f"({len(excluded_subjects)} excluded)")
    if len(excluded_subjects):
        print(f"  Excluded: {excluded_subjects}")

    return rune_data, complete_subjects, excluded_subjects

# LOAD DATA

CSV_PATH_T1 = "data/pgng_anonymized_1st_timepoint.csv"
CSV_PATH_T2 = "data/pgng_tb3_anonymized_2nd_timepoint.csv"
PIT_ID_PATH = "data/PIT_ID_afterexclusion.csv"

# ---- load PIT inclusion list -------------------------------------------
# Only subjects in this list passed PIT task QC; passed into build_rune_data
# so the rune-building loop never processes subjects we don't need.

pit_ids = pd.read_csv(PIT_ID_PATH)["new_ID"].values
print(f"PIT inclusion list: {len(pit_ids)} subjects")

# ---- load both timepoints ----------------------------------------------

print("\nLoading Timepoint 1...")
# subjects_t1_raw = every subject in the CSV, including incomplete ones
# df_t1           = cleaned dataframe, passed to build_rune_data below
actions_t1, rewards_t1, subjects_t1_raw, rune_map_t1, df_t1 = load_real_data_with_df(CSV_PATH_T1)

# build_rune_data first restricts to pit_ids, then drops any remaining
# subjects with malformed rune counts (not exactly 3 runes per robot per block)
# subjects_t1  = only PIT-approved subjects who passed the rune-count check
# excluded_t1  = subjects dropped due to malformed rune counts (logged, not used further)
# rune_data_t1 = nested list aligned to subjects_t1 (not subjects_t1_raw)
rune_data_t1, subjects_t1, excluded_t1 = build_rune_data(df_t1, subjects_t1_raw, pit_ids=pit_ids)

print("\nLoading Timepoint 2...")
actions_t2, rewards_t2, subjects_t2_raw, rune_map_t2, df_t2 = load_real_data_with_df(CSV_PATH_T2)
rune_data_t2, subjects_t2, excluded_t2 = build_rune_data(df_t2, subjects_t2_raw, pit_ids=pit_ids)

# ---- align actions/rewards to complete subjects only -------------------
# actions/rewards were built from subjects_t1_raw (everyone in the CSV),
# but rune_data only contains PIT-approved complete subjects — so we trim
# actions/rewards to match, using subject IDs as the alignment key.

def subset_arrays(actions, rewards, subjects_raw, subjects_complete):
    # For each complete subject, find its row index in the original raw array
    idx = [np.where(subjects_raw == s)[0][0] for s in subjects_complete]
    actions_sub = {rob: actions[rob][idx, :] for rob in robots}  # keep only complete-subject rows
    rewards_sub = {rob: rewards[rob][idx, :] for rob in robots}
    return actions_sub, rewards_sub

# Overwrite with trimmed versions — after this, actions_t1/t2 and
# rune_data_t1/t2 are both aligned to the same subjects_t1/t2 list
actions_t1, rewards_t1 = subset_arrays(actions_t1, rewards_t1, subjects_t1_raw, subjects_t1)
actions_t2, rewards_t2 = subset_arrays(actions_t2, rewards_t2, subjects_t2_raw, subjects_t2)

# ---- sanity check: print rune → robot mappings for first 3 subjects ---
# Verifies that rune letters are correctly mapped to robot types,
# and that each subject has the expected number of runes per robot.

for label, subjects_tp, rune_map in [
    ("Timepoint 1", subjects_t1, rune_map_t1),
    ("Timepoint 2", subjects_t2, rune_map_t2),
]:
    print(f"\n{label} rune → robot mappings (first 3 subjects):")
    for subj in subjects_tp[:3]:

        # Invert the rune → robot dict to robot → [runes] for a readable printout
        robot_to_runes = {}
        for rune, robot in rune_map[subj].items():
            robot_to_runes.setdefault(robot, []).append(rune)  # group rune letters under their robot

        # Format as "GAL: ['A', 'B']  GW: ['C', 'D'] ..."
        mapping_str = "  ".join(
            f"{rob}: {sorted(runes)}" for rob, runes in sorted(robot_to_runes.items())
        )
        print(f"  {subj}: {mapping_str}")

In [ ]:
# MODEL LIKELIHOOD & FITTING

def neg_log_likelihood(params, rune_data_b1, rune_data_b2):
    """
    M7 negative log-likelihood across both blocks for a single subject.

    Model equation (from Samuel & Gili's paper):
        p(go) = (1 - ξ) · σ(β_v · [Q(Go) - Q(NoGo)] + τ_v) + ξ/2

    where:
        σ(·)  = logistic sigmoid = 1 / (1 + exp(-x))
        β_v   = inverse temperature (sensitivity to Q difference); v = win/loss
        τ_v   = go bias; positive = bias toward go, negative = bias toward nogo
        ξ     = lapse rate: probability of a random response (not fitted here, fixed=0)
        η_v   = learning rate α; scales prediction error on each update

    Q-update rule:
        Q(action) ← Q(action) + η_v · [r - Q(action)]
        i.e. Q moves toward the received reward r, at rate η_v

    Win robots  (GW, NGW)  use params: α+, β+, τ+
    Loss robots (GAL, NGAL) use params: α-, β-, τ-

    Each rune has its own independent Q-table, reset to 0.5 at rune start.
    4 robots × 3 runes × 2 blocks = 24 independent Q-traces per subject.

    params       = [α+, α-, β+, β-, τ+, τ-]
    rune_data_bN : dict  robot → {rune → {"actions": (1, n), "rewards": (1, n)}}
    """

    # Unpack the 6 parameters
    alpha_p, alpha_m, beta_p, beta_m, tau_p, tau_m = params

    nll = 0.0   # accumulate negative log-likelihood across all trials

    for block_data in [rune_data_b1, rune_data_b2]:        # iterate over both blocks
        for robot, runes in block_data.items():

            # Select win or loss parameter set based on robot type
            # Win robots: rewarded for correct action → α+, β+, τ+
            # Loss robots: punished for incorrect action → α-, β-, τ-
            alpha, beta, tau = (
                (alpha_p, beta_p, tau_p) if robot in win_robots
                else (alpha_m, beta_m, tau_m)
            )

            for rune, arrays in runes.items():

                a_seq = arrays["actions"][0]   # binary action sequence: 1=go, 0=nogo
                r_seq = arrays["rewards"][0]   # reward sequence: recoded to 1=better outcome, 0=worse outcome 
                # Independent Q-values for go and nogo, reset at the start of each rune.
                # Initialised at 0.5 — neutral starting point, reflecting no prior preference
                # for go or nogo before any feedback has been received for this rune.
                Q = {"go": 0.5, "nogo": 0.5}

                for t in range(len(a_seq)):

                    # ---- compute p(go) using the paper's equation ----

                    q_diff = Q["go"] - Q["nogo"]   # signed value of choosing go over nogo;
                                                   # positive = go is currently preferred,
                                                   # negative = nogo is currently preferred

                    # Core sigmoid term: σ(β · [Q(go) - Q(nogo)] + τ)
                    # β scales how strongly the Q-difference drives choice:
                    #   β → 0 : choice is nearly random regardless of Q-values
                    #   β → ∞ : choice is nearly deterministic (always pick higher-Q action)
                    # τ is an additive go bias independent of learned values:
                    #   τ > 0 : baseline tendency to go even when Q-values are equal
                    #   τ < 0 : baseline tendency to nogo even when Q-values are equal
                    p_go_latent = sigmoid(beta * q_diff + tau)

                    # Full model equation including lapse rate ξ:
                    # p(go) = (1 - ξ) · σ(·) + ξ/2
                    # ξ = 0 here (no lapse term fitted), so p(go) = σ(·)
                    # To add lapse: replace 0.0 with a fitted xi parameter
                    xi   = 0.0
                    p_go = (1 - xi) * p_go_latent + xi / 2

                    # ---- compute trial log-likelihood ----------------

                    a = a_seq[t]   # action taken: 1=go, 0=nogo
                    r = r_seq[t]   # reward received: 1=better outcome, 0=worse outcome

                    # Likelihood of the observed action under the model:
                    # if subject went → probability of that is p_go
                    # if subject did not go → probability of that is 1 - p_go
                    p    = p_go if a == 1 else (1 - p_go)
                    # Accumulate negative log-likelihood; epsilon (1e-12) prevents log(0)
                    # which would occur if p_go is exactly 0 or 1 due to extreme parameters
                    nll -= np.log(p + 1e-12)

                    # ---- Q update (Rescorla-Wagner rule) -------------

                    # Only the chosen action's Q is updated — the unchosen action's Q
                    # stays the same because no information was received about it.
                    # Q(action) ← Q(action) + α · [r - Q(action)]
                    # The prediction error [r - Q] is positive when the outcome was
                    # better than expected (Q moves up) and negative when worse (Q moves down).
                    # α controls how fast Q adapts: α=1 means Q jumps to r immediately;
                    # α→0 means Q barely updates and learning is very slow.
                    if a == 1:
                        Q["go"]   += alpha * (r - Q["go"])
                    else:
                        Q["nogo"] += alpha * (r - Q["nogo"])

    return nll


def fit_m7(rune_data_b1, rune_data_b2, s_idx, n_starts=10):
    """
    Fit M7 to a single subject using L-BFGS-B with multiple random starts.

    Multiple starts are used because the NLL surface can have local minima —
    we run the optimiser from n_starts different random initialisations and
    keep whichever converged to the lowest NLL.

    s_idx is the subject's position in the subject list (0, 1, 2, ...).
    It is used to seed the random number generator so that:
      - each subject gets a different set of starting points (avoids all
        subjects being initialised identically, which would mean the same
        local minimum is found for everyone if data are similar)
      - starting points are reproducible across script runs (same seed →
        same starting points → same result every time you run the code)

    Parameter bounds — set to exclude degenerate edge cases:
        α+, α-  ∈ [0.01, 0.99]   strictly inside (0,1): α=0 means no learning at all
                                  (flat likelihood, optimizer stalls); α=1 means complete
                                  forgetting on every trial (implausible, also unstable)
        β+, β-  ∈ [0.01, 15]     β=0 means purely random choice (unidentifiable with τ);
                                  β>10 already means near-deterministic choice, so 15
                                  gives headroom without allowing unrealistic values
        τ+, τ-  ∈ [-4, 4]        at |τ|=4, sigmoid(4)≈0.98 so bias alone already
                                  drives p(go) to near ceiling/floor; beyond ±4 the
                                  model is essentially ignoring learned Q-values entirely
    """

    bounds = [
        (0.01, 0.99),  (0.01, 0.99),   # α+, α-
        (0.01, 8),    (0.01, 8),      # β+, β-
        (-4,   4),     (-4,   4),       # τ+, τ-
    ]

    # Seed the RNG on s_idx so each subject gets a unique but reproducible
    # sequence of starting points across all n_starts runs.
    # Example: subject 0 → seed 42, subject 1 → seed 43, ..., subject 227 → seed 269.
    # Without this, every subject would use the same 10 starting points,
    # which reduces the effective diversity of the multi-start search.
    rng = np.random.default_rng(42 + s_idx)

    best_res = None

    for _ in range(n_starts):

        # Draw a fresh random starting point for this run.
        # Ranges are chosen to cover the most behaviorally plausible region
        # of parameter space while staying well inside the bounds above.
        # Each call to rng.uniform() advances the RNG state, so successive
        # runs automatically get different starting points for the same subject.
        x0 = [
            rng.uniform(0.01, 0.99),   # α+: anywhere in the valid learning rate range
            rng.uniform(0.01, 0.99),   # α-
            rng.uniform(0.1,  5.0),    # β+: concentrate exploration in the most common range
            rng.uniform(0.1,  5.0),    # β-  (most fitted betas fall between 1–8 in the literature)
            rng.uniform(-2.0, 2.0),    # τ+: moderate bias range; ±2 covers most realistic biases
            rng.uniform(-2.0, 2.0),    # τ-
        ]

        res = minimize(
            neg_log_likelihood, x0,
            args=(rune_data_b1, rune_data_b2),
            bounds=bounds,
            method="L-BFGS-B",   # gradient-based method that respects parameter bounds;
                                  # efficient for smooth NLL surfaces like this one
        )

        # Keep this result only if it converged to a lower NLL than the current best.
        # After all n_starts runs, best_res holds the global minimum found.
        if best_res is None or res.fun < best_res.fun:
            best_res = res

    return best_res.x   # return the 6 best-fitting parameter values: [α+, α-, β+, β-, τ+, τ-]



# FIT M7 TO EACH SUBJECT — TIMEPOINT 1 AND TIMEPOINT 2 SEPARATELY

param_names = ["alpha_plus", "alpha_minus", "beta_plus", "beta_minus", "tau_plus", "tau_minus"]

fitted_t1 = []
fitted_t2 = []

for timepoint, rune_data, subjects_tp, store in [
    (1, rune_data_t1, subjects_t1, fitted_t1),
    (2, rune_data_t2, subjects_t2, fitted_t2),
]:
    print(f"\nFitting Timepoint {timepoint} ({len(subjects_tp)} subjects)...")

    for s_idx, subj in enumerate(subjects_tp):

        # Extract this subject's rune data for each block separately.
        # neg_log_likelihood takes blocks as two separate arguments so it
        # can reset Q-values between blocks — each block is an independent
        # learning episode with fresh Q-tables for every rune.
        subj_rune_b1 = rune_data[s_idx][1]   # block 1: robot → rune → arrays
        subj_rune_b2 = rune_data[s_idx][2]   # block 2: robot → rune → arrays

        # Pass s_idx into fit_m7 so the RNG is seeded differently per subject,
        # giving each subject a unique but reproducible set of starting points
        est = fit_m7(subj_rune_b1, subj_rune_b2, s_idx)
        store.append(est)

        # Progress report every 10 subjects and on the first subject
        if (s_idx + 1) % 10 == 0 or s_idx == 0:
            print(f"  T{timepoint} fitted {s_idx + 1}/{len(subjects_tp)}: {subj}")

# Convert lists of 6-element arrays to (n_subjects, 6) matrices
fitted_t1 = np.array(fitted_t1)
fitted_t2 = np.array(fitted_t2)

# ---- save fitted parameters to CSV ------------------------------------

SAVE_DIR = "data/"

for timepoint, subjects_tp, arr in [
    (1, subjects_t1, fitted_t1),
    (2, subjects_t2, fitted_t2),
]:
    out = pd.DataFrame(arr, columns=param_names)
    out.insert(0, "subject",   subjects_tp)   # subject ID as first column
    out.insert(1, "timepoint", timepoint)     # timepoint label as second column
    fname = f"{SAVE_DIR}fitted_params_timepoint{timepoint}.csv"
    out.to_csv(fname, index=False)
    print(f"Saved {fname}")

# ---- sanity check: medians should be in a plausible range -------------
# Expected rough ranges from the literature for this task:
#   α:  0.1 – 0.5   (moderate learning rates)
#   β:  1.0 – 8.0   (meaningful but not extreme sensitivity)
#   τ:  -2.0 – 2.0  (moderate go/nogo bias)
# If medians are at or near the bounds, that suggests the bounds may be
# too tight and artificially capping parameter estimates for many subjects.
print()
print("Timepoint 1 medians:", np.round(np.median(fitted_t1, axis=0), 3))
print("Timepoint 2 medians:", np.round(np.median(fitted_t2, axis=0), 3))

# ---- boundary pile-up check -------------------------------------------
# Count how many subjects have parameters within 1% of their bounds.
# A large number (>10% of subjects) at any bound suggests that bound
# is constraining the fit and should be reconsidered.
for label, arr in [("T1", fitted_t1), ("T2", fitted_t2)]:
    beta_p_at_bound = np.sum(arr[:, 2] >= 14.9)   # β+ near upper bound of 15
    beta_m_at_bound = np.sum(arr[:, 3] >= 14.9)   # β- near upper bound of 15
    print(f"{label}: {beta_p_at_bound} subjects hit β+ bound, "
          f"{beta_m_at_bound} subjects hit β- bound")



In [ ]:
def load_and_filter(csv_path, rune_data, min_alpha=-1, min_go_rate=0.1, max_go_rate=0.9):
    """
    Load fitted params CSV and filter by:
      1. Learning rate: both alpha+ and alpha- must exceed min_alpha
      2. Go-rate: subject's overall go-rate must be between min_go_rate and max_go_rate
                  — excludes subjects who responded only go or only nogo the entire task

    min_alpha=-1    → no alpha filtering by default
    min_go_rate=0.1 → exclude subjects who went on fewer than 10% of trials
    max_go_rate=0.9 → exclude subjects who went on more than 90% of trials
    """

    df = pd.read_csv(csv_path)
    subjects_arr = df["subject"].values
    fitted_arr   = df[["alpha_plus", "alpha_minus", "beta_plus", "beta_minus",
                        "tau_plus", "tau_minus"]].values

    # ---- mask 1: learning rate filter ----------------------------------

    alpha_mask = (fitted_arr[:, 0] > min_alpha) & (fitted_arr[:, 1] > min_alpha)

    # ---- mask 2: go-rate filter ----------------------------------------

    # For each subject compute their overall go-rate across all trials,
    # all robots, all blocks, all runes
    go_rate_mask = np.zeros(len(subjects_arr), dtype=bool)

    for s_idx in range(len(subjects_arr)):
        all_actions = []

        for block in [1, 2]:
            for rob in robots:
                for rune, arrays in rune_data[s_idx][block][rob].items():
                    all_actions.extend(arrays["actions"][0].tolist())  # 1=go, 0=nogo

        go_rate = np.mean(all_actions)   # proportion of go responses across all trials
        go_rate_mask[s_idx] = (go_rate >= min_go_rate) and (go_rate <= max_go_rate)

        # # Report any subject being excluded by this criterion
        # if not go_rate_mask[s_idx]:
        #     print(f"  EXCLUDED subject index {s_idx} (ID={subjects_arr[s_idx]}): "
        #           f"go-rate={go_rate:.3f} (threshold: [{min_go_rate}, {max_go_rate}])")

    # ---- combine both masks --------------------------------------------

    # Subject must pass BOTH the alpha and go-rate filters to be kept
    mask = alpha_mask & go_rate_mask

    n_excluded_alpha   = (~alpha_mask).sum()
    n_excluded_go_rate = (~go_rate_mask).sum()
    n_excluded_total   = (~mask).sum()

    print(f"  [{csv_path}]")
    print(f"    Excluded by alpha filter:   {n_excluded_alpha}")
    print(f"    Excluded by go-rate filter: {n_excluded_go_rate}")
    print(f"    Excluded total:             {n_excluded_total} / {len(df)}")
    print(f"    Kept:                       {mask.sum()} / {len(df)}")

    rune_data_filtered = [rune_data[i] for i, keep in enumerate(mask) if keep]

    return fitted_arr[mask], subjects_arr[mask], rune_data_filtered


fitted_t1, subjects_t1_fit, rune_data_t1_fit = load_and_filter(
    "data/fitted_params_timepoint1.csv",
    rune_data_t1,
)
fitted_t2, subjects_t2_fit, rune_data_t2_fit = load_and_filter(
    "data/fitted_params_timepoint2.csv",
    rune_data_t2,
)

In [ ]:
# LOOK AT LEARNING RATE DISTRIBUTIONS AND POTENTIAL EXCLUSION THRESHOLDS

print("Timepoint 1 alpha distributions:")
print(f"  alpha_plus  — min={fitted_t1[:,0].min():.4f}  median={np.median(fitted_t1[:,0]):.4f}  max={fitted_t1[:,0].max():.4f}")
print(f"  alpha_minus — min={fitted_t1[:,1].min():.4f}  median={np.median(fitted_t1[:,1]):.4f}  max={fitted_t1[:,1].max():.4f}")

print("\nTimepoint 2 alpha distributions:")
print(f"  alpha_plus  — min={fitted_t2[:,0].min():.4f}  median={np.median(fitted_t2[:,0]):.4f}  max={fitted_t2[:,0].max():.4f}")
print(f"  alpha_minus — min={fitted_t2[:,1].min():.4f}  median={np.median(fitted_t2[:,1]):.4f}  max={fitted_t2[:,1].max():.4f}")

# Also show how many subjects fall below various thresholds
for thresh in [0.001, 0.01, 0.05, 0.1]:
    n_excluded_t1 = ((fitted_t1[:,0] <= thresh) | (fitted_t1[:,1] <= thresh)).sum()
    n_excluded_t2 = ((fitted_t2[:,0] <= thresh) | (fitted_t2[:,1] <= thresh)).sum()
    print(f"\nthreshold={thresh}: would exclude {n_excluded_t1}/{len(fitted_t1)} T1,  {n_excluded_t2}/{len(fitted_t2)} T2")

# Count how many runes have 8, 10, or 12 trials across all subjects
from collections import Counter

lengths = []
for s_idx in range(len(rune_data_t1_fit)):
    for block in [1, 2]:
        for rob in robots:
            for rune, arrays in rune_data_t1_fit[s_idx][block][rob].items():
                lengths.append(arrays["actions"].shape[1])

print(f"\nTrial Counter={Counter(lengths)}")

In [ ]:
# MATCH SUBJECTS ACROSS TIMEPOINTS — KEEP ONLY SUBJECTS PRESENT IN BOTH

# Find subject IDs that appear in both timepoints
# These are the subjects we can track longitudinally
shared_subjects = np.intersect1d(subjects_t1_fit, subjects_t2_fit)
print(f"\nSubjects in T1 only:     {len(subjects_t1_fit)}")
print(f"Subjects in T2 only:     {len(subjects_t2_fit)}")
print(f"Subjects in both (matched): {len(shared_subjects)}")

def subset_to_shared(fitted_arr, subjects_arr, rune_data, shared_subjects):
    """
    Subset fitted parameters, subject IDs, and rune_data to only the
    subjects present in shared_subjects, in the order of shared_subjects.
    
    Reordering to match shared_subjects ensures T1 and T2 arrays are
    in the same subject order — essential for any paired analysis.
    """
    # For each shared subject, find its index in this timepoint's subject array
    # np.where returns the row index where the subject ID matches
    idx = [np.where(subjects_arr == s)[0][0] for s in shared_subjects]

    fitted_sub   = fitted_arr[idx]              # (n_shared, 6) — reordered parameter matrix
    subjects_sub = subjects_arr[idx]            # (n_shared,)   — reordered subject IDs
    rune_sub     = [rune_data[i] for i in idx]  # list of length n_shared — reordered rune dicts

    return fitted_sub, subjects_sub, rune_sub


# Apply to both timepoints — after this, index i in T1 and T2 is the same subject
fitted_t1, subjects_t1_fit, rune_data_t1_fit = subset_to_shared(
    fitted_t1, subjects_t1_fit, rune_data_t1_fit, shared_subjects
)
fitted_t2, subjects_t2_fit, rune_data_t2_fit = subset_to_shared(
    fitted_t2, subjects_t2_fit, rune_data_t2_fit, shared_subjects
)

# Sanity check — these should now be identical arrays
assert np.array_equal(subjects_t1_fit, subjects_t2_fit), \
    "Subject order mismatch between timepoints after matching!"

print(f"\nFinal matched sample: {len(shared_subjects)} subjects")
print(f"Subject order verified — T1 and T2 are aligned.")

In [ ]:
# VISUALISE: PARAMETER ESTIMATES BY TIMEPOINT — REWARDING vs PUNISHING

from scipy import stats

# Each subplot shows one parameter pair (+ and -), split into rewarding (left)
# and punishing (right) halves, with timepoint 1 and 2 plotted within each half.

# Parameter pairs: (plus_col_idx, minus_col_idx, display_label, y_axis_label)
param_plots = [
    (2, 3, "β",  "Outcome sensitivity (β)"),   # beta_plus, beta_minus
    (0, 1, "α",  "Learning rate (α)"),          # alpha_plus, alpha_minus
    (4, 5, "τ",  "Go bias (τ)"),                # tau_plus, tau_minus
]

# Colors matching the figure style
color_reward  = "#2a5fa5"   # blue  — rewarding condition
color_punish  = "#7b1e1e"   # dark red — punishing condition

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
fig.suptitle(
    f"M7 parameter estimates by timepoint\n"
    f"T1: n={len(subjects_t1_fit)}   T2: n={len(subjects_t2_fit)}",
    fontsize=13
)

for ax, (plus_idx, minus_idx, param_label, y_label) in zip(axes, param_plots):

    # ---- extract values for each condition and timepoint ---------------

    # Rewarding condition uses the "plus" parameter (β+, α+, τ+)
    rew_t1 = fitted_t1[:, plus_idx]
    rew_t2 = fitted_t2[:, plus_idx]

    # Punishing condition uses the "minus" parameter (β-, α-, τ-)
    pun_t1 = fitted_t1[:, minus_idx]
    pun_t2 = fitted_t2[:, minus_idx]

    # ---- compute mean and 95% CI (mean ± 1.96 * SEM) ------------------
    # Using SEM-based CI to match the style in the figure

    def mean_ci(vals):
        """Return (mean, lower_err, upper_err) where errs are distances from mean."""
        m   = np.mean(vals)
        sem = stats.sem(vals)
        ci  = 1.96 * sem # 95% confidence interval = 1.96 * SEM for a normal distribution
        return m, ci, ci   # symmetric CI

    rew_t1_m, rew_t1_lo, rew_t1_hi = mean_ci(rew_t1)
    rew_t2_m, rew_t2_lo, rew_t2_hi = mean_ci(rew_t2)
    pun_t1_m, pun_t1_lo, pun_t1_hi = mean_ci(pun_t1)
    pun_t2_m, pun_t2_lo, pun_t2_hi = mean_ci(pun_t2)

    # ---- x positions ---------------------------------------------------
    # Left half: rewarding T1=1, T2=2
    # Right half: punishing T1=4, T2=5  (gap of 1.5 units separates the halves)

    x_rew = [1, 2]
    x_pun = [3.5, 4.5]

    # ---- plot rewarding (left half) ------------------------------------

    ax.errorbar(
        x_rew,
        [rew_t1_m, rew_t2_m],
        yerr=[[rew_t1_lo, rew_t2_lo], [rew_t1_hi, rew_t2_hi]],
        fmt="o",                    # circle markers
        color=color_reward,
        markersize=10,
        linewidth=1.5,              # error bar line width
        capsize=0,                  # no caps on error bars
        label="Rewarding",
        zorder=3,
    )

    # ---- plot punishing (right half) -----------------------------------

    ax.errorbar(
        x_pun,
        [pun_t1_m, pun_t2_m],
        yerr=[[pun_t1_lo, pun_t2_lo], [pun_t1_hi, pun_t2_hi]],
        fmt="o",
        color=color_punish,
        markersize=10,
        linewidth=1.5,
        capsize=0,
        label="Punishing",
        zorder=3,
    )

    # ---- axes formatting -----------------------------------------------

    ax.set_title(f"Parameter: {param_label}", fontsize=12)
    ax.set_ylabel(y_label, fontsize=11)
    ax.set_xlabel("Timepoint", fontsize=11)

    # Set x ticks to show timepoint labels under each dot
    ax.set_xticks([1, 2, 3.5, 4.5])
    ax.set_xticklabels(["1", "2", "1", "2"])
    ax.set_xlim(0.3, 5.2)

    # Remove top and right spines to match clean figure style
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

    # Only add legend to the first subplot to avoid repetition
    if ax == axes[0]:
        ax.legend(frameon=False, fontsize=10)

plt.tight_layout()
plt.savefig(f"{SAVE_DIR}param_estimates_by_timepoint.png", dpi=150)
plt.show()
print("Saved param_estimates_by_timepoint.png")

In [ ]:
# VISUALISE: TIMEPOINT 1 vs TIMEPOINT 2 PARAMETER DISTRIBUTIONS

# 2 rows × 3 columns — one subplot per parameter (α+, α-, β+, β-, τ+, τ-)
fig, axes = plt.subplots(2, 3, figsize=(13, 7))

for i, (ax, name) in enumerate(zip(axes.flat, param_names)):
    # axes.flat iterates subplots left-to-right, top-to-bottom
    # i is the parameter index (0=α+, 1=α-, 2=β+, 3=β-, 4=τ+, 5=τ-)

    # Extract all subjects' estimates for this parameter at each timepoint
    t1_vals = fitted_t1[:, i]   # (n_t1_subjects,)
    t2_vals = fitted_t2[:, i]   # (n_t2_subjects,)

    # Normalise to % of subjects so T1 and T2 are directly comparable
    # despite having different n's — each bar = (count in bin / total) * 100
    w_t1 = np.ones(len(t1_vals)) / len(t1_vals) * 100
    w_t2 = np.ones(len(t2_vals)) / len(t2_vals) * 100

    # Overlapping histograms; alpha=0.6 so both are visible where they overlap
    ax.hist(t1_vals, bins=30, alpha=0.6, color="steelblue",
            edgecolor="white", weights=w_t1)
    ax.hist(t2_vals, bins=30, alpha=0.6, color="darkorange",
            edgecolor="white", weights=w_t2)

    # Vertical reference lines — dashed = median, solid = mean
    # Both are labelled so they appear in the legend with their values
    ax.axvline(np.mean(t1_vals),   color="steelblue",  linestyle="--",  linewidth=1.5,
               label=f"T1 mean={np.mean(t1_vals):.2f}")
    ax.axvline(np.mean(t2_vals),   color="darkorange", linestyle="--",  linewidth=1.5,
               label=f"T2 mean={np.mean(t2_vals):.2f}")

    ax.set_title(name)
    ax.set_xlabel("Estimated value")
    ax.set_ylabel("% of subjects")
    ax.legend(fontsize=8)

plt.suptitle(
    f"M7 parameter estimates — "
    f"Timepoint 1 ({len(subjects_t1_fit)} subjects) vs Timepoint 2 ({len(subjects_t2_fit)} subjects)",
    fontsize=13
)
plt.tight_layout()

SAVE_DIR = "data/"

plt.savefig(f"{SAVE_DIR}fitted_params_timepoint1_vs_timepoint2.png", dpi=150)
plt.show()
print("Saved fitted_params_timepoint1_vs_timepoint2.png")


def compute_actual_go(rune_data, subjects):
    """
    Average go-choices at each within-rune trial position across all
    3 runes × 2 blocks (6 curves per robot per subject).

    Runes have 8, 10, or 12 trials — we pad shorter runes with NaN so all
    curves are length 12, then nanmean so positions beyond a short rune's
    end don't artificially drag the average down.

    Returns
    -------
    mean_pgo : dict  robot → (12,)  grand mean across subjects
    ci_pgo   : dict  robot → (12,)  95% CI half-width (mean ± this value gives the band)
    """
    n          = len(subjects)
    max_trials = 12

    mean_pgo = {}
    ci_pgo   = {}   # renamed from sem_pgo to reflect what it now contains

    for rob in robots:

        subj_means = np.full((n, max_trials), np.nan)

        for s_idx in range(n):
            curves = []

            for block in [1, 2]:
                for rune, arrays in rune_data[s_idx][block][rob].items():
                    seq    = arrays["actions"][0]
                    padded = np.full(max_trials, np.nan)
                    padded[:len(seq)] = seq
                    curves.append(padded)

            subj_means[s_idx] = np.nanmean(curves, axis=0)

        mean_pgo[rob] = np.nanmean(subj_means, axis=0)

        # Per-position n — varies at positions 8–11 where short runes have ended
        n_obs = np.sum(~np.isnan(subj_means), axis=0)
        sem   = np.nanstd(subj_means, axis=0) / np.sqrt(n_obs)

        # 95% CI half-width = 1.96 × SEM (assumes normal sampling distribution,
        # reasonable for n > ~30; for small n you could use scipy.stats.t.ppf instead)
        ci_pgo[rob] = 1.96 * sem

    return mean_pgo, ci_pgo

colors = {
    "GW":   "#1f4e79",  # dark blue
    "NGW":  "#7fb3d5",  # light blue
    "NGAL": "#7b1e1e",  # dark red
    "GAL":  "#e6a0a0",  # light red
}

trial_axis = np.arange(1, 13)   # defined once outside the loop: [1, 2, ..., 12]

fig, axes = plt.subplots(1, 2, figsize=(10, 6), sharey=True)

for ax, TIMEPOINT in zip(axes, [1, 2]):

    rune_data_plot = rune_data_t1_fit if TIMEPOINT == 1 else rune_data_t2_fit
    subjects_plot  = subjects_t1_fit  if TIMEPOINT == 1 else subjects_t2_fit

    mean_pgo, ci_pgo = compute_actual_go(rune_data_plot, subjects_plot)

    ax.set_title(f"Timepoint {TIMEPOINT}  (n={len(subjects_plot)})", fontsize=18)
    ax.set_xlabel("Trial", fontsize=18)
    ax.set_ylim(0.0, 1.0)
    ax.set_xlim(1, 12)
    ax.set_xticks([1, 6, 12])
    ax.axhline(0.5, color="grey", alpha=0.35, linestyle="-", linewidth=3)

    for rob in robots:
        m  = mean_pgo[rob]
        ci = ci_pgo[rob]
        ax.fill_between(trial_axis, m - ci, m + ci, color=colors[rob], alpha=0.25)
        ax.plot(trial_axis, m, color=colors[rob], linewidth=3.5, label=rob)

    if TIMEPOINT == 1:
        ax.set_ylabel("Mean Go Response", fontsize=18)
        ax.legend(frameon=False, title="Robot")

    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.tick_params(axis="both", labelsize=18)

fig.suptitle(
    "Mean ± 95% CI Actual Go-Choices\nTrial position averaged across 3 runes × 2 blocks",
    fontsize=20
)
plt.tight_layout()
plt.savefig(f"{SAVE_DIR}pgo_over_trials_both_timepoints.png", dpi=150)
plt.show()
print("Saved pgo_over_trials_both_timepoints.png")

In [ ]:
def simulate_pgo(fitted_arr, rune_data, subjects):
    """
    For each subject, simulate go-probabilities at each within-rune trial
    position using their fitted M7 parameters and the actual reward sequences
    they experienced. This gives the model's predicted p(go) trajectory.

    Model equation (same as neg_log_likelihood):
        p(go) = (1 - ξ) · σ(β_v · [Q(Go) - Q(NoGo)] + τ_v) + ξ/2
        Q(action) ← Q(action) + α_v · [r - Q(action)]

    Rather than sampling binary choices, we record the deterministic p(go)
    at each trial — this is directly comparable
    to the mean go-choice plots produced by compute_actual_pgo.

    Parameters
    ----------
    fitted_arr : (n_subjects, 6)  columns = [α+, α-, β+, β-, τ+, τ-]
    rune_data  : list of per-subject dicts (aligned to fitted_arr)
    subjects   : array of subject IDs (used only for n)

    Returns
    -------
    mean_pgo : dict  robot → (12,)
    ci_pgo   : dict  robot → (12,)  95% CI half-width
    """

    n          = len(subjects)   # number of subjects
    max_trials = 12              # longest possible rune; shorter runes will be NaN-padded

    mean_pgo = {}   # will store grand mean p(go) per robot across subjects
    ci_pgo   = {}   # will store 95% CI half-width per robot across subjects

    for rob in robots:   # loop over all 4 robot types (GW, NGW, GAL, NGAL)

        # Pre-allocate a (n_subjects × 12) matrix of NaNs
        # Each row = one subject's mean predicted p(go) at each trial position
        # NaN fill means missing positions (short runes) won't contaminate averages
        subj_means = np.full((n, max_trials), np.nan)

        for s_idx in range(n):   # loop over subjects

            # Unpack all 6 fitted parameters for this subject from the parameter matrix
            alpha_p, alpha_m, beta_p, beta_m, tau_p, tau_m = fitted_arr[s_idx]

            # Select the win (rewarding) or loss (punishing) parameter set
            # based on which robot type we're currently simulating:
            # GW/NGW → win robots → use α+, β+, τ+
            # GAL/NGAL → loss robots → use α-, β-, τ-
            alpha, beta, tau = (
                (alpha_p, beta_p, tau_p) if rob in win_robots
                else (alpha_m, beta_m, tau_m)
            )

            curves = []   # collect one p(go) trajectory per rune per block (6 total: 3 runes × 2 blocks)

            for block in [1, 2]:   # loop over both blocks

                for rune, arrays in rune_data[s_idx][block][rob].items():   # loop over the 3 runes for this robot in this block

                    r_seq = arrays["rewards"][0]   # actual reward sequence this subject received (length 8, 10, or 12)
                    n_t   = len(r_seq)             # actual number of trials for this rune

                    # Reset Q-values to neutral at the start of every rune —
                    # each rune is an independent learning episode, same as in fitting
                    Q = {"go": 0.5, "nogo": 0.5}

                    p_gos = []   # store the model's predicted p(go) at each trial within this rune

                    for t in range(n_t):   # loop over trials within this rune

                        # ---- STEP 1: compute p(go) BEFORE updating Q ----
                        # This matches the order in neg_log_likelihood:
                        # predict first, then learn from the outcome

                        # Value advantage of going over not going at this trial
                        q_diff = Q["go"] - Q["nogo"]

                        # Core sigmoid term: σ(β · [Q(go) - Q(nogo)] + τ)
                        # β scales how strongly the Q difference drives the decision
                        # τ is an additive go bias (positive = more go, negative = more nogo)
                        p_go_latent = sigmoid(beta * q_diff + tau)

                        # Full M7 equation including lapse rate ξ:
                        # p(go) = (1 - ξ) · σ(·) + ξ/2
                        # ξ = 0 here (no lapse fitted), so p(go) = σ(·)
                        # ξ/2 would add a floor of random responding if ξ > 0
                        xi   = 0.0
                        p_go = (1 - xi) * p_go_latent + xi / 2

                        p_gos.append(p_go)   # record the predicted probability for this trial

                        # ---- STEP 2: update Q using the ACTUAL action + reward ----
                        # We use what the subject actually did (not a sampled choice)
                        # so the Q-trace follows the subject's real experience —
                        # this is what makes this a posterior predictive check rather
                        # than a pure forward simulation
                        a = arrays["actions"][0][t]   # subject's actual action: 1=go, 0=nogo
                        r = r_seq[t]                  # reward they actually received

                        # Rescorla-Wagner delta rule:
                        # Q(action) ← Q(action) + α · [r - Q(action)]
                        # Only the chosen action's Q value is updated
                        if a == 1:
                            Q["go"]   += alpha * (r - Q["go"])
                        else:
                            Q["nogo"] += alpha * (r - Q["nogo"])

                    # NaN-pad this rune's p(go) trajectory to length 12
                    # so all curves are the same length and can be averaged
                    # positions beyond the rune's actual length stay NaN
                    padded       = np.full(max_trials, np.nan)
                    padded[:n_t] = p_gos
                    curves.append(padded)   # add this rune's trajectory to the collection

            # Average the 6 rune trajectories for this subject at each trial position
            # nanmean ignores NaN so short runes don't contribute zeros to later positions
            subj_means[s_idx] = np.nanmean(curves, axis=0)

        # ---- aggregate across subjects ----------------------------------------

        # Grand mean across all subjects at each of the 12 trial positions
        mean_pgo[rob] = np.nanmean(subj_means, axis=0)

        # Count how many subjects actually contributed at each trial position —
        # this varies at positions 8–11 where only longer runes reach
        # (all 3 lengths contribute at 0–7, 2 lengths at 8–9, only 12-trial runes at 10–11)
        n_obs = np.sum(~np.isnan(subj_means), axis=0)

        # SEM = std / sqrt(n) computed per trial position using the actual n at that position
        sem = np.nanstd(subj_means, axis=0) / np.sqrt(n_obs)

        # 95% CI half-width = 1.96 × SEM
        # The band plotted will be mean ± ci_pgo, i.e. [mean - 1.96*SEM, mean + 1.96*SEM]
        ci_pgo[rob] = 1.96 * sem

    return mean_pgo, ci_pgo


# ---- plot: one subplot per timepoint, side by side ---------------------
trial_axis = np.arange(1, 13)

fig, axes = plt.subplots(1, 2, figsize=(10, 6), sharey=True)

for ax, TIMEPOINT in zip(axes, [1, 2]):

    rune_data_plot = rune_data_t1_fit if TIMEPOINT == 1 else rune_data_t2_fit
    subjects_plot  = subjects_t1_fit  if TIMEPOINT == 1 else subjects_t2_fit
    fitted_plot    = fitted_t1        if TIMEPOINT == 1 else fitted_t2

    mean_pgo, ci_pgo = simulate_pgo(fitted_plot, rune_data_plot, subjects_plot)

    ax.set_title(f"Timepoint {TIMEPOINT}  (n={len(subjects_plot)})", fontsize=18)
    ax.set_xlabel("Trial", fontsize=18)
    ax.set_ylim(0.0, 1.0)
    ax.set_xlim(1, 12)
    ax.set_xticks([1, 6, 12])
    ax.axhline(0.5, color="grey", alpha=0.35, linestyle="-", linewidth=3)

    for rob in robots:
        m  = mean_pgo[rob]
        ci = ci_pgo[rob]
        ax.fill_between(trial_axis, m - ci, m + ci, color=colors[rob], alpha=0.25)
        ax.plot(trial_axis, m, color=colors[rob], linewidth=3.5, label=rob)

    if TIMEPOINT == 1:
        ax.set_ylabel("Mean P(Go)", fontsize=18)
        ax.legend(frameon=False, title="Robot")

    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.tick_params(axis="both", labelsize=18)

fig.suptitle(
    "Mean ± 95% CI Predicted Go-Probability\n"
    "Trial position averaged across 3 runes × 2 blocks",
    fontsize=20
)
plt.tight_layout()
plt.savefig(f"{SAVE_DIR}ppc_pgo_over_trials_both_timepoints.png", dpi=150)
plt.show()
print("Saved ppc_pgo_over_trials_both_timepoints.png")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 6), sharey=True)

for ax, TIMEPOINT in zip(axes, [1, 2]):

    rune_data_plot = rune_data_t1_fit if TIMEPOINT == 1 else rune_data_t2_fit
    subjects_plot  = subjects_t1_fit  if TIMEPOINT == 1 else subjects_t2_fit
    fitted_plot    = fitted_t1        if TIMEPOINT == 1 else fitted_t2

    # Actual data
    mean_actual, ci_actual = compute_actual_go(rune_data_plot, subjects_plot)
    # Model predictions
    mean_pred, ci_pred = simulate_pgo(fitted_plot, rune_data_plot, subjects_plot)

    for rob in robots:
        # Actual — solid line
        ax.plot(trial_axis, mean_actual[rob], color=colors[rob],
                linewidth=2.5, label=rob)
        ax.fill_between(trial_axis,
                        mean_actual[rob] - ci_actual[rob],
                        mean_actual[rob] + ci_actual[rob],
                        color=colors[rob], alpha=0.15)

        # Predicted — dashed line, same color
        ax.plot(trial_axis, mean_pred[rob], color=colors[rob],
                linewidth=2.5, linestyle="--")

    ax.set_title(f"Timepoint {TIMEPOINT}  (n={len(subjects_plot)})", fontsize=14)
    ax.set_xlabel("Trial", fontsize=12)
    ax.set_xlim(1, 12)
    ax.set_xticks([1, 6, 12])
    ax.set_ylim(0.0, 1.0)
    ax.axhline(0.5, color="grey", alpha=0.35, linestyle="-", linewidth=2)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.tick_params(axis="both", labelsize=12)

    if TIMEPOINT == 1:
        ax.set_ylabel("Mean Go Response", fontsize=12)
        # Custom legend: solid = actual, dashed = predicted
        from matplotlib.lines import Line2D
        legend_elements = [
            Line2D([0], [0], color="black", linewidth=2,   linestyle="-",  label="Actual"),
            Line2D([0], [0], color="black", linewidth=2,   linestyle="--", label="Predicted"),
        ]
        ax.legend(handles=legend_elements, frameon=False)

fig.suptitle(
    "Actual (solid) vs Model-Predicted (dashed) Go-Choices\n"
    "Trial position averaged across 3 runes × 2 blocks",
    fontsize=16
)
plt.tight_layout()
plt.savefig(f"{SAVE_DIR}ppc_overlay_both_timepoints.png", dpi=150)
plt.show()
print("Saved ppc_overlay_both_timepoints.png")